In [ ]:
print("notebook connected")

In [ ]:
from pathlib import Path
import os 
import numpy as np
import tensorflow as tf 
import tifffile 



In [ ]:


DATA_ROOT = Path("/home/tdeibert/Data")
IMAGE_ROOT = DATA_ROOT / "Control/Extract_Two"

Image = IMAGE_ROOT / "Untitled.tif"

assert Image.exists(), f"Image not found at {Image}"
MODEL_PATH = DATA_ROOT / "/home/tdeibert/Data/Training_Data_v2_512px1/Focus/models/unet_droplet_nucleus_best_1.2.h5"
assert MODEL_PATH.exists(), f"Model not found at {MODEL_PATH}"
model= tf.keras.models.load_model(MODEL_PATH)
img = tifffile.memmap(Image)  

In [ ]:
#### Metadata Interrogation#### 

with tifffile.TiffFile(Image) as tif:
    print("Number of series:", len(tif.series))
    
    for i, s in enumerate(tif.series):
        print(f"\nSeries {i}")
        print("  shape:", s.shape)
        print("  axes: ", s.axes)
        print("  dtype:", s.dtype)


In [ ]:
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)

In [ ]:
import numpy as np

t = 0
z = 0

plane = img[t, z, :, :, :]          # shape: (3, 392, 396)
plane = np.moveaxis(plane, 0, -1)   # shape: (392, 396, 3)

print("Plane shape:", plane.shape)

In [ ]:
import tensorflow as tf

plane_resized = tf.image.resize(plane, (512, 512)).numpy()
print(plane_resized.shape)

In [ ]:
x = plane_resized.astype(np.float32)
x = x / x.max()

In [ ]:
x = x[np.newaxis, ...]
print("Input to model:", x.shape)

In [ ]:
pred = model.predict(x, verbose=0)
print("Prediction shape:", pred.shape)

In [ ]:
label_map = np.argmax(pred[0], axis=-1).astype(np.uint8)
print("Unique labels:", np.unique(label_map))

In [ ]:
import numpy as np
import tensorflow as tf


t = 9
z = 15

# extract one TZ plane with all 3 channels
plane = img[t, z, :, :, :]          # (3, 392, 396)
plane = np.moveaxis(plane, 0, -1)   # (392, 396, 3)

# pad to model input size
def pad_to_512(x):
    h, w, c = x.shape
    out = np.zeros((512, 512, c), dtype=x.dtype)
    out[:h, :w, :] = x
    return out

plane_512 = pad_to_512(plane)

# normalize
x = plane_512.astype(np.float32)
x = x / x.max()

# add batch dimension
x = x[np.newaxis, ...]

print("Model input shape:", model.input_shape)
print("Input to model:", x.shape)

# predict
pred = model.predict(x, verbose=0)
print("Prediction shape:", pred.shape)

# convert to class labels
label_map = np.argmax(pred[0], axis=-1).astype(np.uint8)
print("Unique labels:", np.unique(label_map))

In [ ]:
plane = img[t, z, :, :, :]
plane = np.moveaxis(plane, 0, -1)   # (Y, X, C)

In [ ]:
npc_prob = pred[0, ..., 2]   # adjust if needed
npc_mask = npc_prob > 0.5

In [ ]:
membrane = plane[..., 0]

In [ ]:
from skimage.measure import label, regionprops

lab = label(npc_mask)
props = regionprops(lab)

largest = max(props, key=lambda r: r.area)
cy, cx = largest.centroid

In [ ]:
import numpy as np
from skimage.measure import label, regionprops

def get_main_mask_centroid(mask):
    lab = label(mask)
    props = regionprops(lab)
    if not props:
        return None
    largest = max(props, key=lambda r: r.area)
    return largest.centroid

def radial_profile(image, center, n_angles=180, max_radius=100):
    cy, cx = center
    angles = np.linspace(0, 2*np.pi, n_angles, endpoint=False)
    
    profiles = []

    for theta in angles:
        values = []
        for r in range(max_radius):
            x = int(round(cx + r * np.cos(theta)))
            y = int(round(cy + r * np.sin(theta)))
            
            if x < 0 or x >= image.shape[1] or y < 0 or y >= image.shape[0]:
                break
            
            values.append(image[y, x])
        profiles.append(values)

    return profiles, angles

In [ ]:
# raw membrane channel
membrane = plane[..., 0]

# predicted NPC class map
npc_prob = pred[0, ..., 2]   # change if class order differs

# threshold
npc_mask = npc_prob > 0.5

# if padded, crop back to original image size
npc_mask = npc_mask[:membrane.shape[0], :membrane.shape[1]]

center = get_main_mask_centroid(npc_mask)

profiles, angles = radial_profile(
    membrane,
    center=center,
    n_angles=180,
    max_radius=150
)

In [ ]:
def radial_profile_array(image, center, n_angles=180, max_radius=100):
    cy, cx = center
    angles = np.linspace(0, 2*np.pi, n_angles, endpoint=False)
    out = np.full((n_angles, max_radius), np.nan, dtype=float)

    for i, theta in enumerate(angles):
        for r in range(max_radius):
            x = int(round(cx + r * np.cos(theta)))
            y = int(round(cy + r * np.sin(theta)))

            if x < 0 or x >= image.shape[1] or y < 0 or y >= image.shape[0]:
                break

            out[i, r] = image[y, x]

    return out, angles

In [ ]:
radial_mat, angles = radial_profile_array(
    membrane,
    center=center,
    n_angles=180,
    max_radius=150
)

In [ ]:
import numpy as np
import napari
from skimage.measure import label, regionprops

# ----------------------------
# 1. choose one T/Z plane
# ----------------------------
t = 9
z = 15

# full plane from raw image: (C, Y, X) -> (Y, X, C)
plane = img[t, z, :, :, :]
plane = np.moveaxis(plane, 0, -1)

# raw membrane channel for measurement
membrane = plane[..., 0]

# ----------------------------
# 2. prepare model input
# ----------------------------
def pad_to_512(x):
    h, w, c = x.shape
    out = np.zeros((512, 512, c), dtype=x.dtype)
    out[:h, :w, :] = x
    return out

plane_512 = pad_to_512(plane)

x = plane_512.astype(np.float32)
x = x / x.max()
x = x[np.newaxis, ...]

# predict
pred = model.predict(x, verbose=0)

# ----------------------------
# 3. extract NPC class map
# ----------------------------
# change index if NPC is not class 2
npc_prob = pred[0, ..., 2]

# crop back to original image size
npc_prob = npc_prob[:membrane.shape[0], :membrane.shape[1]]

# threshold to binary mask
npc_mask = npc_prob > 0.5

# ----------------------------
# 4. keep largest connected object
# ----------------------------
lab = label(npc_mask)
props = regionprops(lab)

if len(props) == 0:
    raise ValueError("No NPC object detected in this plane.")

largest = max(props, key=lambda r: r.area)
main_mask = lab == largest.label
cy, cx = largest.centroid

print(f"Centroid: (y={cy:.2f}, x={cx:.2f})")

# ----------------------------
# 5. build centroid point layer
# ----------------------------
centroid_points = np.array([[cy, cx]])

# ----------------------------
# 6. build radial sweep lines
# ----------------------------
def make_radial_lines(center, image_shape, n_angles=36, max_radius=150):
    cy, cx = center
    lines = []

    for theta in np.linspace(0, 2*np.pi, n_angles, endpoint=False):
        x2 = cx + max_radius * np.cos(theta)
        y2 = cy + max_radius * np.sin(theta)

        # clip endpoint to image bounds
        x2 = np.clip(x2, 0, image_shape[1] - 1)
        y2 = np.clip(y2, 0, image_shape[0] - 1)

        line = np.array([
            [cy, cx],
            [y2, x2]
        ])
        lines.append(line)

    return lines

radial_lines = make_radial_lines(
    center=(cy, cx),
    image_shape=membrane.shape,
    n_angles=180,
    max_radius=150
)

# ----------------------------
# 7. open in napari
# ----------------------------
viewer = napari.Viewer()

viewer.add_image(
    membrane,
    name="membrane_ch0"
)

viewer.add_image(
    npc_prob,
    name="npc_probability",
    opacity=0.4
)

viewer.add_labels(
    main_mask.astype(np.uint8),
    name="npc_main_mask",
    opacity=0.5
)

viewer.add_points(
    centroid_points,
    name="centroid",
    size=8
)

viewer.add_shapes(
    radial_lines,
    shape_type="line",
    name="radial_lines",
    edge_width=2
)

napari.run()

In [ ]:
import numpy as np
import pandas as pd

def sample_radial_profiles(image, center, n_angles=180, max_radius=150):
    """
    Sample intensities from a 2D image along radial lines.

    Parameters
    ----------
    image : 2D np.ndarray
        Membrane image (Y, X)
    center : tuple
        (cy, cx)
    n_angles : int
        Number of angular sweeps
    max_radius : int
        Maximum radial distance in pixels

    Returns
    -------
    radial_mat : np.ndarray
        Shape (n_angles, max_radius), values are sampled intensities
    angles_deg : np.ndarray
        Angles in degrees
    """
    cy, cx = center
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    angles_deg = np.degrees(angles)

    radial_mat = np.full((n_angles, max_radius), np.nan, dtype=np.float32)

    for i, theta in enumerate(angles):
        for r in range(max_radius):
            x = int(round(cx + r * np.cos(theta)))
            y = int(round(cy + r * np.sin(theta)))

            if x < 0 or x >= image.shape[1] or y < 0 or y >= image.shape[0]:
                break

            radial_mat[i, r] = image[y, x]

    return radial_mat, angles_deg

In [ ]:
import numpy as np

def sample_radial_profiles_vectorized(image, cx, cy, n_angles=180, max_radius=150):
    """
    Sample image intensities along radial lines from center (cx, cy).

    Parameters
    ----------
    image : 2D numpy array
        Image to sample from.
    cx, cy : float
        Center point in pixel coordinates.
    n_angles : int
        Number of radial directions.
    max_radius : int
        Maximum radius in pixels.

    Returns
    -------
    radial_values : ndarray, shape (n_angles, max_radius)
        Sampled intensities. Rows are angles, columns are radii.
    angles : ndarray
        Angles in radians.
    radii : ndarray
        Radius values in pixels.
    """
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    radii = np.arange(max_radius)

    R, T = np.meshgrid(radii, angles)

    x = cx + R * np.cos(T)
    y = cy + R * np.sin(T)

    x_idx = np.rint(x).astype(int)
    y_idx = np.rint(y).astype(int)

    h, w = image.shape[:2]

    valid = (
        (x_idx >= 0) & (x_idx < w) &
        (y_idx >= 0) & (y_idx < h)
    )

    radial_values = np.full((n_angles, max_radius), np.nan, dtype=float)
    radial_values[valid] = image[y_idx[valid], x_idx[valid]]

    return radial_values, angles, radii

In [ ]:
from skimage.measure import regionprops

props = regionprops(labeled_mask)

In [ ]:
for prop in props:
    cy, cx = prop.centroid  # NOTE: (row, col) = (y, x)

    radial_mat, angles, radii = sample_radial_profiles_vectorized(
        image=plane,
        cx=cx,
        cy=cy,
        n_angles=180,
        max_radius=150
    )

In [ ]:
radial_mat, angles, radii = sample_radial_profiles_vectorized(
    image=plane,
    cx=nucleus_cx,
    cy=nucleus_cy,
    n_angles=180,
    max_radius=150
)

print(radial_mat.shape)
# (180, 150)

In [ ]:
mask_values = np.zeros((n_angles, max_radius), dtype=bool)
mask_values[valid] = droplet_mask[y_idx[valid], x_idx[valid]]

In [ ]:
radial_values_masked = radial_values.copy()
radial_values_masked[~mask_values] = np.nan

In [ ]:
radial_mat, angles_deg = sample_radial_profiles(
    image=membrane,
    center=(cy, cx),
    n_angles=180,
    max_radius=150
)

print(radial_mat.shape)

In [ ]:
mean_profile = np.nanmean(radial_mat, axis=0)
peak_by_angle = np.nanmax(radial_mat, axis=1)
peak_radius_by_angle = np.nanargmax(radial_mat, axis=1)

In [ ]:
def radial_matrix_to_long_df(radial_mat, angles_deg, t=None, z=None):
    records = []

    for angle_idx, angle_deg in enumerate(angles_deg):
        for radius in range(radial_mat.shape[1]):
            intensity = radial_mat[angle_idx, radius]

            records.append({
                "t": t,
                "z": z,
                "angle_idx": angle_idx,
                "angle_deg": angle_deg,
                "radius_px": radius,
                "intensity": intensity
            })

    return pd.DataFrame(records)

radial_df = radial_matrix_to_long_df(
    radial_mat=radial_mat,
    angles_deg=angles_deg,
    t=t,
    z=z
)

radial_df.head()

In [ ]:
def summarize_radial_profiles(radial_mat, angles_deg, t=None, z=None):
    mean_profile = np.nanmean(radial_mat, axis=0)
    peak_by_angle = np.nanmax(radial_mat, axis=1)
    peak_radius_by_angle = np.nanargmax(radial_mat, axis=1)

    summary = pd.DataFrame({
        "t": [t],
        "z": [z],
        "mean_intensity": [np.nanmean(radial_mat)],
        "max_intensity": [np.nanmax(radial_mat)],
        "mean_peak_intensity": [np.nanmean(peak_by_angle)],
        "sd_peak_intensity": [np.nanstd(peak_by_angle)],
        "mean_peak_radius": [np.nanmean(peak_radius_by_angle)],
        "sd_peak_radius": [np.nanstd(peak_radius_by_angle)]
    })

    angle_summary = pd.DataFrame({
        "t": t,
        "z": z,
        "angle_deg": angles_deg,
        "peak_intensity": peak_by_angle,
        "peak_radius_px": peak_radius_by_angle
    })

    radial_summary = pd.DataFrame({
        "t": t,
        "z": z,
        "radius_px": np.arange(len(mean_profile)),
        "mean_intensity": mean_profile
    })

    return summary, angle_summary, radial_summary

In [ ]:
summary_df, angle_df, radius_df = summarize_radial_profiles(
    radial_mat,
    angles_deg,
    t=t,
    z=z
)

print(summary_df)

In [ ]:
compaction_score = np.nanstd(peak_by_angle) / np.nanmean(peak_by_angle)
print("Compaction score:", compaction_score)

In [ ]:
ray_mean = np.nanmean(radial_mat, axis=1)
ray_peak = np.nanmax(radial_mat, axis=1)

peak_enrichment = ray_peak / ray_mean
print("Mean peak enrichment:", np.nanmean(peak_enrichment))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(np.arange(radial_mat.shape[1]), np.nanmean(radial_mat, axis=0))
plt.xlabel("Radius (px)")
plt.ylabel("Mean membrane intensity")
plt.title("Mean radial profile")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(radial_mat, aspect='auto', interpolation='nearest')
plt.xlabel("Radius (px)")
plt.ylabel("Angle index")
plt.title("Radial sweep intensity matrix")
plt.colorbar(label="Membrane intensity")
plt.tight_layout()
plt.show()

In [ ]:
radial_mat, angles_deg = sample_radial_profiles(
    image=membrane,
    center=(cy, cx),
    n_angles=180,
    max_radius=150
)

summary_df, angle_df, radius_df = summarize_radial_profiles(
    radial_mat,
    angles_deg,
    t=t,
    z=z
)

print(summary_df)

compaction_score = np.nanstd(np.nanmax(radial_mat, axis=1)) / np.nanmean(np.nanmax(radial_mat, axis=1))
print("Compaction score:", compaction_score)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from skimage.measure import label, regionprops

# --------------------------------------------------
# helpers
# --------------------------------------------------
def pad_to_512(x):
    h, w, c = x.shape
    out = np.zeros((512, 512, c), dtype=x.dtype)
    out[:h, :w, :] = x
    return out

def predict_npc_mask(plane_yxc, model, npc_class_idx=2, threshold=0.5):
    """
    plane_yxc: (Y, X, C)
    returns:
        npc_prob: (Y, X)
        npc_mask: (Y, X) boolean
    """
    y0, x0, _ = plane_yxc.shape

    plane_512 = pad_to_512(plane_yxc)
    x = plane_512.astype(np.float32)

    # match this to your training normalization if different
    if x.max() > 0:
        x = x / x.max()

    x = x[np.newaxis, ...]  # (1, 512, 512, 3)

    pred = model.predict(x, verbose=0)
    npc_prob = pred[0, ..., npc_class_idx]

    # crop back to original image size
    npc_prob = npc_prob[:y0, :x0]
    npc_mask = npc_prob > threshold

    return npc_prob, npc_mask

def largest_object_props(mask):
    """
    returns largest connected component mask + props
    """
    lab = label(mask)
    props = regionprops(lab)

    if len(props) == 0:
        return None, None

    largest = max(props, key=lambda r: r.area)
    main_mask = lab == largest.label
    return main_mask, largest

def sample_radial_profiles(image, center, n_angles=180, max_radius=150):
    """
    image: 2D array (Y, X)
    center: (cy, cx)
    returns radial matrix: (n_angles, max_radius)
    """
    cy, cx = center
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    radial_mat = np.full((n_angles, max_radius), np.nan, dtype=np.float32)

    for i, theta in enumerate(angles):
        for r in range(max_radius):
            x = int(round(cx + r * np.cos(theta)))
            y = int(round(cy + r * np.sin(theta)))

            if x < 0 or x >= image.shape[1] or y < 0 or y >= image.shape[0]:
                break

            radial_mat[i, r] = image[y, x]

    return radial_mat

def compaction_score_from_radial(radial_mat):
    """
    Compaction = CV of peak intensity across angles
    """
    peak_by_angle = np.nanmax(radial_mat, axis=1)

    if np.all(np.isnan(peak_by_angle)):
        return np.nan

    mean_peak = np.nanmean(peak_by_angle)
    sd_peak = np.nanstd(peak_by_angle)

    if mean_peak == 0 or np.isnan(mean_peak):
        return np.nan

    return sd_peak / mean_peak

In [ ]:
# --------------------------------------------------
# main loop
# --------------------------------------------------
results = []

T, Z, C, Y, X = img.shape
print("Image shape:", img.shape)

for t in range(T):
    best_area = -1
    best_record = None

    for z in range(Z):
        # extract one plane: (C, Y, X) -> (Y, X, C)
        plane = img[t, z, :, :, :]
        plane = np.moveaxis(plane, 0, -1)

        membrane = plane[..., 0]

        # predict NPC mask
        npc_prob, npc_mask = predict_npc_mask(
            plane_yxc=plane,
            model=model,
            npc_class_idx=2,   # change if needed
            threshold=0.5
        )

        main_mask, props = largest_object_props(npc_mask)

        if props is None:
            continue

        area = props.area
        cy, cx = props.centroid

        # choose max-area z slice for this time point
        if area > best_area:
            radial_mat = sample_radial_profiles(
                image=membrane,
                center=(cy, cx),
                n_angles=180,
                max_radius=150
            )

            compaction = compaction_score_from_radial(radial_mat)

            best_area = area
            best_record = {
                "t": t,
                "best_z": z,
                "npc_area_px": area,
                "centroid_y": cy,
                "centroid_x": cx,
                "compaction_score": compaction
            }

    if best_record is not None:
        results.append(best_record)

time_df = pd.DataFrame(results)
time_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(time_df["t"], time_df["compaction_score"], marker="o")
plt.xlabel("Time point")
plt.ylabel("Membrane compaction score")
plt.title("Membrane compaction over time\n(using max NPC area Z-slice per time point)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(time_df["t"], time_df["npc_area_px"], marker="o")
plt.xlabel("Time point")
plt.ylabel("Max NPC mask area (px)")
plt.title("Largest NPC mask area by time point")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

def compute_radial_metrics(radial_mat, inner_baseline_px=20, outer_offset_px=5, outer_width_px=60):
    """
    Compute 4 biologically useful metrics from a radial intensity matrix.

    Parameters
    ----------
    radial_mat : np.ndarray
        Shape = (n_angles, max_radius)
    inner_baseline_px : int
        Number of inner pixels used to estimate baseline intensity
    outer_offset_px : int
        Number of pixels to skip after the envelope peak before measuring outer region
    outer_width_px : int
        Width of the outer region window in pixels

    Returns
    -------
    dict
        Dictionary of summary metrics
    """
    mean_profile = np.nanmean(radial_mat, axis=0)

    # 1. Envelope radius
    envelope_radius = int(np.nanargmax(mean_profile))

    # 2. Envelope sharpness
    inner_end = min(inner_baseline_px, radial_mat.shape[1])
    baseline = np.nanmean(mean_profile[:inner_end])
    peak_val = np.nanmax(mean_profile)

    if baseline == 0 or np.isnan(baseline):
        envelope_sharpness = np.nan
    else:
        envelope_sharpness = peak_val / baseline

    # Define outer region
    outer_start = envelope_radius + outer_offset_px
    outer_end = min(outer_start + outer_width_px, radial_mat.shape[1])

    if outer_start >= radial_mat.shape[1]:
        external_region = np.full((radial_mat.shape[0], 1), np.nan)
    else:
        external_region = radial_mat[:, outer_start:outer_end]

    # 3. External heterogeneity
    ext_mean = np.nanmean(external_region)
    ext_sd = np.nanstd(external_region)

    if ext_mean == 0 or np.isnan(ext_mean):
        external_heterogeneity = np.nan
    else:
        external_heterogeneity = ext_sd / ext_mean

    # 4. Angular asymmetry
    angle_signal = np.nanmean(external_region, axis=1)
    angle_mean = np.nanmean(angle_signal)
    angle_sd = np.nanstd(angle_signal)

    if angle_mean == 0 or np.isnan(angle_mean):
        angular_asymmetry = np.nan
    else:
        angular_asymmetry = angle_sd / angle_mean

    return {
        "envelope_radius_px": envelope_radius,
        "envelope_peak_intensity": peak_val,
        "inner_baseline_intensity": baseline,
        "envelope_sharpness": envelope_sharpness,
        "outer_start_px": outer_start,
        "outer_end_px": outer_end,
        "external_heterogeneity": external_heterogeneity,
        "angular_asymmetry": angular_asymmetry
    }

In [ ]:
metrics = compute_radial_metrics(radial_mat)
print(metrics)

In [ ]:
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops

results = []

T, Z, C, Y, X = img.shape

for t in range(T):
    best_area = -1
    best_record = None

    for z in range(Z):
        # (C, Y, X) -> (Y, X, C)
        plane = img[t, z, :, :, :]
        plane = np.moveaxis(plane, 0, -1)

        membrane = plane[..., 0]

        # model prediction
        npc_prob, npc_mask = predict_npc_mask(
            plane_yxc=plane,
            model=model,
            npc_class_idx=2,   # change if needed
            threshold=0.5
        )

        main_mask, props = largest_object_props(npc_mask)

        if props is None:
            continue

        area = props.area
        cy, cx = props.centroid

        if area > best_area:
            radial_mat = sample_radial_profiles(
                image=membrane,
                center=(cy, cx),
                n_angles=180,
                max_radius=150
            )

            metric_dict = compute_radial_metrics(
                radial_mat,
                inner_baseline_px=20,
                outer_offset_px=5,
                outer_width_px=60
            )

            best_area = area
            best_record = {
                "t": t,
                "best_z": z,
                "npc_area_px": area,
                "centroid_y": cy,
                "centroid_x": cx,
                **metric_dict
            }

    if best_record is not None:
        results.append(best_record)

metrics_df = pd.DataFrame(results)
metrics_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].plot(metrics_df["t"], metrics_df["envelope_radius_px"], marker="o")
axes[0, 0].set_title("Envelope radius over time")
axes[0, 0].set_xlabel("Time point")
axes[0, 0].set_ylabel("Radius (px)")

axes[0, 1].plot(metrics_df["t"], metrics_df["envelope_sharpness"], marker="o")
axes[0, 1].set_title("Envelope sharpness over time")
axes[0, 1].set_xlabel("Time point")
axes[0, 1].set_ylabel("Peak / baseline")

axes[1, 0].plot(metrics_df["t"], metrics_df["external_heterogeneity"], marker="o")
axes[1, 0].set_title("External heterogeneity over time")
axes[1, 0].set_xlabel("Time point")
axes[1, 0].set_ylabel("CV")

axes[1, 1].plot(metrics_df["t"], metrics_df["angular_asymmetry"], marker="o")
axes[1, 1].set_title("Angular asymmetry over time")
axes[1, 1].set_xlabel("Time point")
axes[1, 1].set_ylabel("CV")

plt.tight_layout()
plt.show()

Large Image Segmentation 

In [ ]:
DATA_ROOT = Path("C:/Users/cowboy/OneDrive/Documents/Unviversity of Alabama")
IMAGE_ROOT = DATA_ROOT / "Nuclear_Scaling/Data_Sets/Control/Extract_3"

Image = IMAGE_ROOT /"Untitled.tif"
assert Image.exists(), f"Image not found at {Image}"
img = tifffile.imread(Image)

In [ ]:
print(img.shape)
print(img.dtype)
plane = img[t, z, :, :, :]
plane = np.moveaxis(plane, 0, -1)

In [ ]:
plane[..., 0]
plane[..., 1]
plane[..., 2]

In [ ]:
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops

def get_valid_objects(mask, min_area_px=100):
    lab = label(mask)
    props = regionprops(lab)

    valid = []
    for prop in props:
        if prop.area < min_area_px:
            continue
        valid.append(prop)

    return lab, valid

In [ ]:
def measure_single_object(
    membrane_image,
    labeled_mask,
    prop,
    t,
    z,
    object_id,
    n_angles=180,
    max_radius=150
):
    cy, cx = prop.centroid
    area_px = prop.area

    radial_mat = sample_radial_profiles(
        image=membrane_image,
        center=(cy, cx),
        n_angles=n_angles,
        max_radius=max_radius
    )

    metric_dict = compute_radial_metrics(
        radial_mat,
        inner_baseline_px=20,
        outer_offset_px=5,
        outer_width_px=60
    )

    return {
        "t": t,
        "z": z,
        "object_id": object_id,
        "centroid_y": cy,
        "centroid_x": cx,
        "npc_cross_section_area_px": area_px,
        **metric_dict
    }

In [ ]:
def measure_plane_objects(
    plane_yxc,
    model,
    t,
    z,
    npc_class_idx=2,
    threshold=0.5,
    min_area_px=100,
    n_angles=180,
    max_radius=150
):
    membrane = plane_yxc[..., 0]

    npc_prob, npc_mask = predict_npc_mask(
        plane_yxc=plane_yxc,
        model=model,
        npc_class_idx=npc_class_idx,
        threshold=threshold
    )

    labeled_mask, props = get_valid_objects(
        npc_mask,
        min_area_px=min_area_px
    )

    plane_results = []

    for i, prop in enumerate(props):
        result = measure_single_object(
            membrane_image=membrane,
            labeled_mask=labeled_mask,
            prop=prop,
            t=t,
            z=z,
            object_id=i,
            n_angles=n_angles,
            max_radius=max_radius
        )
        plane_results.extend(result)

    return plane_results, npc_prob, npc_mask, labeled_mask

In [ ]:
from skimage.measure import label, regionprops
import numpy as np

def measure_plane_objects(
    plane_yxc,
    model,
    t,
    z,
    npc_class_idx=2,
    threshold=0.5,
    min_area_px=100,
    n_angles=180,
    max_radius=150
):
    """
    Measure radial features for each detected nucleus/object in one YXC plane.
    """

    # --- model prediction ---
    pred = model.predict(plane_yxc[None, ...], verbose=0)[0]   # (Y, X, n_classes)
    npc_prob = pred[..., npc_class_idx]
    npc_mask = npc_prob > threshold

    # --- label objects ---
    labeled_mask = label(npc_mask)
    props = regionprops(labeled_mask)

    plane_results = []

    # choose the channel you want to sample
    # adjust this index to match your real biology
    intensity_image = plane_yxc[..., npc_class_idx]

    for obj_id, prop in enumerate(props, start=1):
        if prop.area < min_area_px:
            continue

        # regionprops centroid is (y, x)
        cy, cx = prop.centroid

        # --- vectorized radial sampling ---
        radial_mat, angles, radii = sample_radial_profiles_vectorized(
            image=intensity_image,
            cx=cx,
            cy=cy,
            n_angles=n_angles,
            max_radius=max_radius
        )

        # optional: restrict to this object only
        # object_mask = labeled_mask == prop.label
        # apply mask sampling here if needed

        # --- example summary metrics ---
        peak_intensity = np.nanmax(radial_mat, axis=1)
        peak_radius = np.nanargmax(radial_mat, axis=1)

        # store one row per angle
        for angle_idx, angle_val in enumerate(angles):
            plane_results.extend({
                "t": t,
                "z": z,
                "object_id": prop.label,
                "centroid_x": cx,
                "centroid_y": cy,
                "angle_deg": np.degrees(angle_val),
                "peak_intensity": peak_intensity[angle_idx],
                "peak_radius_px": peak_radius[angle_idx]
            })

    return plane_results, npc_prob, npc_mask, labeled_mask

In [ ]:
import numpy as np

def predict_npc_mask_tiled(
    plane_yxc,
    model,
    npc_class_idx=2,
    threshold=0.5,
    tile_size=512
):
    """
    Run tiled prediction on a large (Y, X, C) image.

    Returns
    -------
    npc_prob : (Y, X) float array
    npc_mask : (Y, X) bool array
    """
    Y, X, C = plane_yxc.shape

    npc_prob_sum = np.zeros((Y, X), dtype=np.float32)
    npc_prob_count = np.zeros((Y, X), dtype=np.float32)

    for y0 in range(0, Y, tile_size):
        for x0 in range(0, X, tile_size):
            y1 = min(y0 + tile_size, Y)
            x1 = min(x0 + tile_size, X)

            patch = plane_yxc[y0:y1, x0:x1, :]   # smaller edge patches possible

            # pad edge patches up to 512x512
            patch_h, patch_w, _ = patch.shape
            patch_pad = np.zeros((tile_size, tile_size, C), dtype=patch.dtype)
            patch_pad[:patch_h, :patch_w, :] = patch

            # normalize like training
            x = patch_pad.astype(np.float32)
            if x.max() > 0:
                x = x / x.max()

            x = x[np.newaxis, ...]   # (1, 512, 512, 3)

            pred = model.predict(x, verbose=0)[0, ..., npc_class_idx]

            # crop prediction back to real patch size
            pred_crop = pred[:patch_h, :patch_w]

            npc_prob_sum[y0:y1, x0:x1] += pred_crop
            npc_prob_count[y0:y1, x0:x1] += 1

    npc_prob = npc_prob_sum / np.maximum(npc_prob_count, 1)
    npc_mask = npc_prob > threshold

    return npc_prob, npc_mask

In [ ]:
def measure_plane_objects(
    plane_yxc,
    model,
    t,
    z,
    npc_class_idx=2,
    threshold=0.5,
    min_area_px=100,
    n_angles=180,
    max_radius=150
):
    membrane = plane_yxc[..., 0]

    npc_prob, npc_mask = predict_npc_mask_tiled(
        plane_yxc=plane_yxc,
        model=model,
        npc_class_idx=npc_class_idx,
        threshold=threshold,
        tile_size=512
    )

    labeled_mask, props = get_valid_objects(
        npc_mask,
        min_area_px=min_area_px
    )

    plane_results = []

    for i, prop in enumerate(props):
        result = measure_single_object(
            membrane_image=membrane,
            labeled_mask=labeled_mask,
            prop=prop,
            t=t,
            z=z,
            object_id=i,
            n_angles=n_angles,
            max_radius=max_radius
        )
        plane_results.extend(result)

    return plane_results, npc_prob, npc_mask, labeled_mask

In [ ]:
#### Vectorized #### 

def measure_single_object(
    membrane_image,
    labeled_mask,
    prop,
    t,
    z,
    object_id,
    n_angles=180,
    max_radius=150
):
    cy, cx = prop.centroid
    object_mask = labeled_mask == prop.label

    radial_mat, angles, radii = sample_radial_profiles_vectorized(
        image=membrane_image,
        cx=cx,
        cy=cy,
        n_angles=n_angles,
        max_radius=max_radius,
        mask=object_mask
    )

    peak_intensity = np.full(n_angles, np.nan, dtype=float)
    peak_radius = np.full(n_angles, np.nan, dtype=float)

    valid_rows = np.any(~np.isnan(radial_mat), axis=1)

    peak_intensity[valid_rows] = np.nanmax(radial_mat[valid_rows], axis=1)

    valid_idx = np.where(valid_rows)[0]
    for idx in valid_idx:
        peak_radius[idx] = np.nanargmax(radial_mat[idx])

    rows = []
    for i in range(n_angles):
        rows.append({
            "t": t,
            "z": z,
            "object_id": object_id,
            "label": prop.label,
            "centroid_x": cx,
            "centroid_y": cy,
            "area_px": prop.area,
            "angle_deg": np.degrees(angles[i]),
            "peak_intensity": peak_intensity[i],
            "peak_radius_px": peak_radius[i]
        })

    return rows

In [ ]:
t = 9
z = 15

plane = img[t, z, :, :, :]
plane = np.moveaxis(plane, 0, -1)

npc_prob, npc_mask = predict_npc_mask_tiled(
    plane_yxc=plane,
    model=model,
    npc_class_idx=2,
    threshold=0.5,
    tile_size=512
)

print(npc_prob.shape, npc_mask.shape)
print(npc_mask.dtype)

In [ ]:
import time

start = time.time()

# load image
t0 = time.time()
# code here
print("Load step:", time.time() - t0)

# segmentation or mask prep
t0 = time.time()
# code here
print("Mask prep:", time.time() - t0)

# radial sweep
t0 = time.time()
# code here
print("Radial sweep:", time.time() - t0)

print("Total:", time.time() - start)

In [ ]:
from scipy.ndimage import binary_fill_holes

def get_object_areas(labeled_mask, prop):
    obj_mask = labeled_mask == prop.label
    filled_mask = binary_fill_holes(obj_mask)

    return {
        "npc_mask_area_px": int(obj_mask.sum()),
        "npc_filled_area_px": int(filled_mask.sum())
    }

In [ ]:
all_results = []

T, Z, C, Y, X = img.shape

for t in range(T):
    for z in range(Z):
        plane = img[t, z, :, :, :]
        plane = np.moveaxis(plane, 0, -1)   # (Y, X, C)

        plane_results, npc_prob, npc_mask, labeled_mask = measure_plane_objects(
            plane_yxc=plane,
            model=model,
            t=t,
            z=z,
            npc_class_idx=2,
            threshold=0.5,
            min_area_px=100,
            n_angles=180,
            max_radius=150
        )

        all_results.extend(plane_results)

results_df = pd.DataFrame(all_results)
print(results_df.head())

In [ ]:
time_start_min = t_index * frame_duration_min
effective_time_min = time_start_min + tile_order * tile_duration_min

In [ ]:
effective_time_min = t_index * 6 + tile_order * 1 + 0.5

In [ ]:
tile_height = Y / n_rows
tile_width  = X / n_cols

In [ ]:
tile_row = int(cy // tile_height)
tile_col = int(cx // tile_width)

In [ ]:
def serpentine_tile_order(tile_row, tile_col, n_cols):
    if tile_row % 2 == 0:
        # even row: left -> right
        return tile_row * n_cols + tile_col
    else:
        # odd row: right -> left
        return tile_row * n_cols + (n_cols - 1 - tile_col)

In [ ]:
def effective_time_minutes(
    t_index,
    tile_row,
    tile_col,
    n_cols,
    frame_duration_min=6,
    tile_duration_min=1,
    use_midpoint=True
):
    tile_order = serpentine_tile_order(tile_row, tile_col, n_cols)
    offset = tile_order * tile_duration_min

    if use_midpoint:
        offset += tile_duration_min / 2

    return t_index * frame_duration_min + offset

In [ ]:
def assign_tile_and_time(
    centroid_y,
    centroid_x,
    image_shape,
    t_index,
    n_rows,
    n_cols,
    frame_duration_min=6,
    tile_duration_min=1
):
    Y, X = image_shape
    tile_h = Y / n_rows
    tile_w = X / n_cols

    tile_row = int(centroid_y // tile_h)
    tile_col = int(centroid_x // tile_w)

    tile_row = min(max(tile_row, 0), n_rows - 1)
    tile_col = min(max(tile_col, 0), n_cols - 1)

    if tile_row % 2 == 0:
        tile_order = tile_row * n_cols + tile_col
    else:
        tile_order = tile_row * n_cols + (n_cols - 1 - tile_col)

    tile_offset_min = tile_order * tile_duration_min + tile_duration_min / 2
    effective_time_min = t_index * frame_duration_min + tile_offset_min

    return {
        "tile_row": tile_row,
        "tile_col": tile_col,
        "tile_order": tile_order,
        "tile_offset_min": tile_offset_min,
        "effective_time_min": effective_time_min
    }

In [ ]:
from scipy.ndimage import binary_fill_holes

def measure_single_object(
    membrane_image,
    labeled_mask,
    prop,
    t,
    z,
    object_id,
    n_angles=180,
    max_radius=150
):
    cy, cx = prop.centroid

    obj_mask = labeled_mask == prop.label
    filled_mask = binary_fill_holes(obj_mask)

    npc_mask_area_px = int(obj_mask.sum())
    npc_filled_area_px = int(filled_mask.sum())

    radial_mat = sample_radial_profiles(
        image=membrane_image,
        center=(cy, cx),
        n_angles=n_angles,
        max_radius=max_radius
    )

    metric_dict = compute_radial_metrics(
        radial_mat,
        inner_baseline_px=20,
        outer_offset_px=5,
        outer_width_px=60
    )
    time_info = assign_tile_and_time(
    centroid_y=cy,
    centroid_x=cx,
    image_shape=membrane_image.shape,
    t_index=t,
    n_rows=2,   # example
    n_cols=3    # example
)

    return {
        "t": t,
        "z": z,
        "object_id": object_id,
        "centroid_y": cy,
        "centroid_x": cx,
        "npc_mask_area_px": npc_mask_area_px,
        "npc_filled_area_px": npc_filled_area_px,
        **metric_dict
    }

In [ ]:
print(all_results.columns.tolist())
print(all_results.head())

In [ ]:
time_summary_df = (
    angle_df
    .groupby("t", as_index=False)
    .agg(
        n_objects=("object_id", "count"),
        mean_mask_area_px=("npc_mask_area_px", "mean"),
        mean_filled_area_px=("npc_filled_area_px", "mean"),
        median_filled_area_px=("npc_filled_area_px", "median"),
        mean_envelope_radius_px=("envelope_radius_px", "mean"),
        mean_envelope_sharpness=("envelope_sharpness", "mean"),
        mean_external_heterogeneity=("external_heterogeneity", "mean"),
        mean_angular_asymmetry=("angular_asymmetry", "mean")
    )
)

print(time_summary_df)